# 02 - ChArUco 深度 scale / bias 校准

RGB 中检测 ChArUco 角点，以已知板几何和 `solvePnP` 计算 `D_true`，从 aligned depth 提取
patch median `D_meas`，拟合 `D_corr = a * D_meas + b`。

本 notebook 已按提供的 calib.io A3 ChArUco 图设置：

- 纸张：A3 横向，420 x 297 mm（纸张尺寸仅用于打印检查，不参与 `solvePnP`）
- 棋盘：水平 10 格、垂直 7 格，即 `squares_x=10, squares_y=7`
- Checker size：35 mm，即 `square_length_m=0.035`
- Marker size：26 mm，即 `marker_length_m=0.026`
- 字典：按原实验要求使用 `cv2.aruco.DICT_5X5_100`

> 图片页脚只写了 `AruCo DICT_5X5`，没有显示 `_50/_100/_250/_1000` 容量后缀。
> 当前采用原 requirement 中明确指定的 `DICT_5X5_100`。若检测不到 marker，应回到生成该板的
> calib.io 项目确认完整 dictionary 名称，而不是修改几何尺寸。


> **运行方式**：从上到下逐个运行。深度数据内部统一使用米（m），统计结果同时显示毫米（mm）。
> 实时采集 cell 需要连接 RealSense D435/D435i。


## Block 1：导入库和参数设置

检查 `cv2.aruco`，并设置 ChArUco 尺寸、过滤阈值和输出目录。


In [ ]:
import json
import time
from datetime import datetime, timezone
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

if not hasattr(cv2, "aruco"):
    raise ImportError("当前 OpenCV 没有 cv2.aruco；请安装 opencv-contrib-python。")
try:
    import pyrealsense2 as rs
except ImportError as exc:
    raise ImportError("缺少 pyrealsense2。") from exc

def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "cam_calibration").is_dir():
            return candidate
    raise RuntimeError("无法定位项目根目录；请从 Xinyu 项目内启动 Jupyter。")

PROJECT_ROOT = find_project_root()
CALIBRATION_ROOT = PROJECT_ROOT / "cam_calibration"
INPUT_DIR = CALIBRATION_ROOT / "inputs"
OUTPUT_ROOT = CALIBRATION_ROOT / "outputs"
SHARED_DIR = OUTPUT_ROOT / "shared"
CONFIG_DIR = CALIBRATION_ROOT / "config"
for directory in (INPUT_DIR, OUTPUT_ROOT, SHARED_DIR, CONFIG_DIR):
    directory.mkdir(parents=True, exist_ok=True)

width, height, fps = 640, 480, 30
patch_size = 7
min_charuco_corners = 6
edge_margin_px = 12
max_abs_initial_error_m = 0.08
warmup_frames = 30
output_dir = OUTPUT_ROOT / "02_charuco_calibration"
output_dir.mkdir(parents=True, exist_ok=True)

# 提供的 calib.io A3 ChArUco board 参数。
paper_size_mm = (420, 297)  # A3 横向；仅作打印比例检查
dictionary_id = cv2.aruco.DICT_5X5_100
dictionary_name = "DICT_5X5_100"
squares_x = 10
squares_y = 7
square_length_m = 0.035
marker_length_m = 0.026

if marker_length_m >= square_length_m:
    raise ValueError("marker_length_m 必须小于 square_length_m。")

board_width_m = squares_x * square_length_m
board_height_m = squares_y * square_length_m
print(f"ChArUco: {squares_x} x {squares_y} squares")
print(f"Checker={square_length_m*1000:.1f} mm, marker={marker_length_m*1000:.1f} mm")
print(f"棋盘格区域理论尺寸: {board_width_m*1000:.1f} x {board_height_m*1000:.1f} mm")
print(f"纸张尺寸: {paper_size_mm[0]} x {paper_size_mm[1]} mm (A3 横向)")
print("Dictionary:", dictionary_name)
print("注意：图片仅标注 DICT_5X5；若 marker 检测失败，请核对完整字典容量后缀。")


## Block 2：启动 RealSense 并获取 color 内参

depth 对齐到 color；构建 OpenCV 相机矩阵与畸变参数。


In [ ]:
pipeline = rs.pipeline()
config = rs.config()
config.enable_stream(rs.stream.color, width, height, rs.format.bgr8, fps)
config.enable_stream(rs.stream.depth, width, height, rs.format.z16, fps)
profile = pipeline.start(config)
align = rs.align(rs.stream.color)
for _ in range(warmup_frames):
    pipeline.wait_for_frames()

depth_scale = float(profile.get_device().first_depth_sensor().get_depth_scale())
intr = profile.get_stream(rs.stream.color).as_video_stream_profile().get_intrinsics()
camera_matrix = np.array([[intr.fx, 0, intr.ppx], [0, intr.fy, intr.ppy], [0, 0, 1]], dtype=np.float64)
rs_coeffs = np.asarray(intr.coeffs, dtype=np.float64)
dist_coeffs = rs_coeffs if np.any(np.abs(rs_coeffs) > 1e-12) else np.zeros(5)
intrinsics = {"width": intr.width, "height": intr.height, "fx": intr.fx, "fy": intr.fy,
              "cx": intr.ppx, "cy": intr.ppy, "model": str(intr.model),
              "coeffs": rs_coeffs.tolist()}
print("camera_matrix:\n", camera_matrix)
print("dist_coeffs:", dist_coeffs)
if not np.any(dist_coeffs):
    print("提醒：当前使用零畸变系数。RealSense color stream 常已校正，但请核对设备模型。")


## Block 3：创建 ChArUco board

兼容新旧 OpenCV API，并读取每个 ChArUco 棋盘角点的三维坐标。


In [ ]:
aruco_dict = cv2.aruco.getPredefinedDictionary(dictionary_id)
try:
    board = cv2.aruco.CharucoBoard((squares_x, squares_y), square_length_m, marker_length_m, aruco_dict)
except (AttributeError, TypeError):
    board = cv2.aruco.CharucoBoard_create(squares_x, squares_y, square_length_m, marker_length_m, aruco_dict)

try:
    board_corners_3d = np.asarray(board.getChessboardCorners(), dtype=np.float32)
except AttributeError:
    board_corners_3d = np.asarray(board.chessboardCorners, dtype=np.float32)

try:
    detector_params = cv2.aruco.DetectorParameters()
except AttributeError:
    detector_params = cv2.aruco.DetectorParameters_create()

print("ChArUco 角点总数:", len(board_corners_3d))


## Block 4：实时预览和检测 ChArUco

定义版本兼容的检测函数并预览一帧。检测图像会保存到输出目录。


In [ ]:
def detect_charuco(color_bgr):
    gray = cv2.cvtColor(color_bgr, cv2.COLOR_BGR2GRAY)
    marker_corners, marker_ids, rejected = cv2.aruco.detectMarkers(
        gray, aruco_dict, parameters=detector_params
    )
    charuco_corners, charuco_ids = None, None
    if marker_ids is not None and len(marker_ids) > 0:
        _, charuco_corners, charuco_ids = cv2.aruco.interpolateCornersCharuco(
            marker_corners, marker_ids, gray, board, camera_matrix, dist_coeffs
        )
    vis = color_bgr.copy()
    if marker_ids is not None:
        cv2.aruco.drawDetectedMarkers(vis, marker_corners, marker_ids)
    if charuco_ids is not None:
        cv2.aruco.drawDetectedCornersCharuco(vis, charuco_corners, charuco_ids)
    return charuco_corners, charuco_ids, vis

frames = align.process(pipeline.wait_for_frames())
preview_color = np.asanyarray(frames.get_color_frame().get_data())
corners, ids, preview_vis = detect_charuco(preview_color)
print("检测到 ChArUco corners:", 0 if ids is None else len(ids))
plt.figure(figsize=(11, 7))
plt.imshow(cv2.cvtColor(preview_vis, cv2.COLOR_BGR2RGB)); plt.axis("off"); plt.show()
cv2.imwrite(str(output_dir / "charuco_preview.png"), preview_vis)


## Block 5：采集多个位姿的数据

运行后出现 OpenCV 窗口：`s` 保存当前有效样本，`q` 结束。每个样本保存 RGB、aligned depth、角点、位姿和深度。


In [ ]:
def patch_median_depth(depth_raw, px, py, scale, size):
    half = size // 2
    if px-half < 0 or py-half < 0 or px+half >= depth_raw.shape[1] or py+half >= depth_raw.shape[0]:
        return np.nan
    patch = depth_raw[py-half:py+half+1, px-half:px+half+1]
    valid = patch[patch > 0].astype(np.float64) * scale
    return float(np.median(valid)) if valid.size else np.nan

def estimate_sample(color_bgr, depth_raw):
    charuco_corners, charuco_ids, vis = detect_charuco(color_bgr)
    if charuco_ids is None or len(charuco_ids) < min_charuco_corners:
        return None, vis
    ids_flat = charuco_ids.reshape(-1).astype(int)
    image_points = charuco_corners.reshape(-1, 2).astype(np.float32)
    object_points = board_corners_3d[ids_flat].astype(np.float32)
    ok, rvec, tvec = cv2.solvePnP(object_points, image_points, camera_matrix, dist_coeffs)
    if not ok:
        return None, vis
    rotation, _ = cv2.Rodrigues(rvec)
    camera_points = (rotation @ object_points.T + tvec.reshape(3, 1)).T
    d_true = camera_points[:, 2]
    d_meas = np.array([
        patch_median_depth(depth_raw, int(round(u)), int(round(v)), depth_scale, patch_size)
        for u, v in image_points
    ])
    sample = dict(corners_px=image_points, ids=ids_flat, rvec=rvec.reshape(3),
                  tvec=tvec.reshape(3), D_true=d_true, D_meas=d_meas)
    return sample, vis

samples = []
print("移动相机/标定板以覆盖多个距离和姿态；按 s 保存，按 q 结束。")
while True:
    frames = align.process(pipeline.wait_for_frames())
    color_frame, depth_frame = frames.get_color_frame(), frames.get_depth_frame()
    if not color_frame or not depth_frame:
        continue
    color = np.asanyarray(color_frame.get_data())
    depth_raw = np.asanyarray(depth_frame.get_data())
    sample, vis = estimate_sample(color, depth_raw)
    count = 0 if sample is None else len(sample["ids"])
    cv2.putText(vis, f"corners={count}, saved={len(samples)} | s=save q=quit",
                (10, 28), cv2.FONT_HERSHEY_SIMPLEX, .65, (0, 255, 0), 2)
    cv2.imshow("ChArUco capture", vis)
    key = cv2.waitKey(1) & 0xFF
    if key == ord("s"):
        if sample is None:
            print("当前帧有效角点不足，未保存。")
            continue
        idx = len(samples)
        sample["color_path"] = str(output_dir / f"sample_{idx:03d}_color.png")
        sample["depth_path"] = str(output_dir / f"sample_{idx:03d}_depth_raw.npy")
        cv2.imwrite(sample["color_path"], color)
        np.save(sample["depth_path"], depth_raw)
        samples.append(sample)
        print(f"保存样本 {idx}，角点 {count} 个。")
    elif key == ord("q"):
        break
cv2.destroyAllWindows()
pipeline.stop()
print("采集结束，样本数:", len(samples))


## Block 6：检查 `D_true`

展开每个样本的 board pose 和角点 camera Z，确认单位与距离范围合理。


In [ ]:
if not samples:
    raise RuntimeError("没有样本。请重新运行采集 block，并至少保存多个距离/姿态。")
for i, sample in enumerate(samples):
    print(f"sample {i}: tvec={sample['tvec']} m, "
          f"D_true=[{np.min(sample['D_true']):.3f}, {np.max(sample['D_true']):.3f}] m")


## Block 7：构造并过滤回归数据

过滤无效 depth、图像边缘和初始绝对误差过大的角点。


In [ ]:
rows = []
for sample_idx, sample in enumerate(samples):
    for corner_id, (uv, d_true, d_meas) in enumerate(
        zip(sample["corners_px"], sample["D_true"], sample["D_meas"])
    ):
        u, v = uv
        rows.append(dict(sample_idx=sample_idx, corner_id=int(sample["ids"][corner_id]),
                         px=float(u), py=float(v), D_true_m=float(d_true), D_meas_m=float(d_meas)))
all_df = pd.DataFrame(rows)
valid_mask = (
    np.isfinite(all_df["D_meas_m"]) & (all_df["D_meas_m"] > 0) &
    np.isfinite(all_df["D_true_m"]) & (all_df["D_true_m"] > 0) &
    (all_df["px"] >= edge_margin_px) & (all_df["px"] < width-edge_margin_px) &
    (all_df["py"] >= edge_margin_px) & (all_df["py"] < height-edge_margin_px) &
    ((all_df["D_meas_m"] - all_df["D_true_m"]).abs() <= max_abs_initial_error_m)
)
regression_df = all_df.loc[valid_mask].copy().reset_index(drop=True)
print(f"有效样本: {len(regression_df)} / {len(all_df)}")
if len(regression_df) < 20:
    print("警告：样本较少。建议覆盖多个距离、板位置和倾角。")


## Block 8：拟合线性模型

使用 `numpy.linalg.lstsq` 拟合 `D_corr = a*D_meas+b`。


In [ ]:
if len(regression_df) < 2:
    raise RuntimeError("有效回归点不足。")
D_meas = regression_df["D_meas_m"].to_numpy()
D_true = regression_df["D_true_m"].to_numpy()
design = np.column_stack([D_meas, np.ones_like(D_meas)])
a, b = np.linalg.lstsq(design, D_true, rcond=None)[0]
D_corr = a * D_meas + b
regression_df["D_corr_m"] = D_corr
print(f"D_corr = {a:.9f} * D_meas + {b:.9f} [m]")


## Block 9：评估修正效果

输出误差统计并绘制修正前后散点图、直方图和误差随真实距离变化。


In [ ]:
def error_metrics(error):
    error = np.asarray(error)
    return {"mean_mm": error.mean()*1000, "std_mm": error.std()*1000,
            "MAE_mm": np.mean(np.abs(error))*1000,
            "RMSE_mm": np.sqrt(np.mean(error**2))*1000,
            "max_abs_mm": np.max(np.abs(error))*1000}

err_before = D_meas - D_true
err_after = D_corr - D_true
metrics_before = error_metrics(err_before)
metrics_after = error_metrics(err_after)
metrics_df = pd.DataFrame([metrics_before, metrics_after], index=["before", "after"])
display(metrics_df)

lo, hi = min(D_true.min(), D_meas.min(), D_corr.min()), max(D_true.max(), D_meas.max(), D_corr.max())
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes[0,0].scatter(D_true, D_meas, s=12, alpha=.6); axes[0,0].plot([lo,hi],[lo,hi],"k--")
axes[0,0].set(xlabel="D_true [m]", ylabel="D_meas [m]", title="Before")
axes[0,1].scatter(D_true, D_corr, s=12, alpha=.6); axes[0,1].plot([lo,hi],[lo,hi],"k--")
axes[0,1].set(xlabel="D_true [m]", ylabel="D_corr [m]", title="After")
axes[0,2].axis("off")
axes[1,0].hist(err_before*1000, bins=30); axes[1,0].set(title="Error before", xlabel="Error [mm]")
axes[1,1].hist(err_after*1000, bins=30); axes[1,1].set(title="Error after", xlabel="Error [mm]")
axes[1,2].scatter(D_true, err_before*1000, s=12, label="before", alpha=.6)
axes[1,2].scatter(D_true, err_after*1000, s=12, label="after", alpha=.6)
axes[1,2].axhline(0, color="k", ls="--"); axes[1,2].set(xlabel="D_true [m]", ylabel="Error [mm]"); axes[1,2].legend()
for ax in axes.flat: ax.grid(True, alpha=.3)
fig.tight_layout(); fig.savefig(output_dir / "calibration_evaluation.png", dpi=150); plt.show()


## Block 10：保存校准参数

保存标准接口 `depth_calibration_linear.json`、CSV、NPZ 和每帧元数据。


In [ ]:
calibration = {
    "model": "D_corr = a * D_meas + b", "a": float(a), "b": float(b),
    "units": "meter", "date_utc": datetime.now(timezone.utc).isoformat(),
    "board": {"dictionary_id": int(dictionary_id), "dictionary_name": dictionary_name,
              "squares_x": squares_x, "squares_y": squares_y,
              "square_length_m": square_length_m, "marker_length_m": marker_length_m,
              "paper_size_mm": list(paper_size_mm),
              "board_size_m": [board_width_m, board_height_m]},
    "camera_intrinsics": intrinsics, "depth_scale": depth_scale,
    "metrics_before": metrics_before, "metrics_after": metrics_after,
    "valid_points": len(regression_df),
}
with open(output_dir / "depth_calibration_linear.json", "w", encoding="utf-8") as f:
    json.dump(calibration, f, indent=2, ensure_ascii=False)
# 复制一份到统一路径，方便 03/04 默认读取。
with open(SHARED_DIR / "depth_calibration_linear.json", "w", encoding="utf-8") as f:
    json.dump(calibration, f, indent=2, ensure_ascii=False)
regression_df.to_csv(output_dir / "charuco_regression_points.csv", index=False)
np.savez_compressed(output_dir / "charuco_regression_points.npz",
                    D_meas_m=D_meas, D_true_m=D_true, D_corr_m=D_corr,
                    err_before_m=err_before, err_after_m=err_after)
metrics_df.to_csv(output_dir / "calibration_metrics.csv")
with open(SHARED_DIR / "camera_intrinsics.json", "w", encoding="utf-8") as f:
    json.dump(intrinsics, f, indent=2, ensure_ascii=False)
print("校准参数已保存:", (SHARED_DIR / "depth_calibration_linear.json").resolve())


## Block 11：结论提示

根据 RMSE 改善比例判断是否存在可由全局线性模型解释的 scale/bias。


In [ ]:
rmse_before = metrics_before["RMSE_mm"]
rmse_after = metrics_after["RMSE_mm"]
improvement = 1 - rmse_after / rmse_before if rmse_before > 0 else 0
print(f"RMSE: {rmse_before:.3f} -> {rmse_after:.3f} mm，改善 {improvement:.1%}")
if improvement >= 0.20:
    print("结论：存在系统性 depth scale/bias，可在后续 pipeline 中使用 D_corr。")
else:
    print("结论：全局线性修正改善不明显；误差更像局部噪声、材质、边缘或 depth-color 对齐误差。")
